# Interactive Emotion Classification Web App

This Colab notebook hosts an interactive web application designed for **Emotion Classification**. It serves as an extension of a project focused on comparing the performance of various emotion classification models.

## Key Features:
- **Model Comparison**: Visualize and compare predictions from different pre-trained models (RoBERTa, DistilBERT, Mistral, Phi-3).
- **Interactive Text Input**: Enter custom text to get real-time emotion predictions from selected models.
- **Batch Analysis**: Upload a CSV file with multiple texts and optional true labels to perform batch predictions and evaluate model performance with metrics like accuracy, precision, recall, and F1-score.
- **Performance Visualization**: See predicted and true emotion distributions for batch analysis.

## Sections Summary:
1.  **Dependency Installation**: Installs necessary libraries like `streamlit`, `torch`, `transformers`, `matplotlib`, `seaborn`, `pandas`, and `pyngrok`.
2.  **`app.py` Creation**: Writes the Streamlit application code, which includes:
    *   UI elements for model selection, text input, and CSV upload.
    *   Functions for loading different pre-trained emotion classification models (BERT-like and LLMs).
    *   Prediction logic for both single text input and batch processing.
    *   Visualization of prediction probabilities and performance metrics.
3.  **Run Application in Localhost**: Sets up `pyngrok` to create a public URL for the Streamlit app, allowing interaction with the web application hosted in Colab.

##Dependencies

In [ ]:
!pip install streamlit torch transformers matplotlib seaborn pandas pyngrok --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 68.6 MB/s eta 0:00:00


##Main App Code

In [ ]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from sklearn.metrics import (
    precision_score, recall_score, f1_score, matthews_corrcoef,
    confusion_matrix
)
from sklearn.manifold import TSNE
import io

st.set_page_config(page_title="Interactive Emotion Classification", layout="wide")
st.title("Interactive Emotion Classification")
st.write("Compare emotion predictions across models.")

# ---------------- DEVICE INDICATOR (SIDEBAR) ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_label = "🟢 GPU (CUDA)" if torch.cuda.is_available() else "🔴 CPU"
st.sidebar.write(f"**Device:** {device_label}")

# ---------------- CONSTANTS ----------------
EMOTIONS = ["sadness", "joy", "love", "anger", "fear", "surprise"]

EMOTION_COLORS = {
    "sadness":  "#5B8DB8",
    "joy":      "#F9C74F",
    "love":     "#F94144",
    "anger":    "#F3722C",
    "fear":     "#8B5CF6",
    "surprise": "#43AA8B",
}

# HF MODELS
HF_MODELS = {
    "RoBERTa (GoEmotions)":  "Sam200309/roberta_goemotions",
    "DistilBERT (GoEmotions)":"Sam200309/distilbert_goemotions",
    "RoBERTa (DAIR-AI)":     "Sam200309/roberta_dair",
    "DistilBERT (DAIR-AI)":  "Sam200309/distilbert_dair"
}

# GoEmotions 28→6 label mapping ----------------
# Maps each of the 28 GoEmotions labels to one of the 6 target emotions.
# Labels not listed here are unmapped and will be grouped into the closest
# category by the aggregation logic below.
GOEMOTIONS_TO_6 = {
    # JOY
    "joy": "joy",
    "amusement": "joy",
    "excitement": "joy",
    "gratitude": "joy",
    "relief": "joy",
    "pride": "joy",
    "optimism": "joy",
    "admiration": "joy",
    "approval": "joy",

    # LOVE
    "love": "love",
    "caring": "love",
    "desire": "love",

    # SADNESS
    "sadness": "sadness",
    "grief": "sadness",
    "disappointment": "sadness",
    "remorse": "sadness",

    # ANGER
    "anger": "anger",
    "annoyance": "anger",

    # FEAR
    "fear": "fear",
    "nervousness": "fear",

    # SURPRISE
    "surprise": "surprise",
    "realization": "surprise",

    # OTHER
    "neutral":     None,
    "curiosity":   None,
    "confusion":   None,
    "disapproval": None,
    "disgust":     None,
    "embarrassment": None,
}

def aggregate_goemotions(probs, raw_labels):
    """
    Collapse 28-class GoEmotions probabilities into the 6 target emotions
    by summing probabilities that share the same target label.
    'neutral' (unmapped) probability is distributed proportionally.
    """
    scores = {e: 0.0 for e in EMOTIONS}
    neutral_prob = 0.0

    for prob, label in zip(probs, raw_labels):
        target = GOEMOTIONS_TO_6.get(label)
        if target is None:
            neutral_prob += prob
        else:
            scores[target] += prob

    # Distribute neutral probability proportionally among the 6 emotions
    total = sum(scores.values())
    if total > 0 and neutral_prob > 0:
        for e in EMOTIONS:
            scores[e] += neutral_prob * (scores[e] / total)

    result = np.array([scores[e] for e in EMOTIONS], dtype=np.float32)
    # Re-normalise to sum to 1 (floating point safety)
    result /= result.sum()
    return result, EMOTIONS


# Model cards shown in sidebar expanders
MODEL_CARDS = {
    "RoBERTa (GoEmotions)":       {"Architecture": "RoBERTa-base", "Training data": "GoEmotions (58k Reddit comments)", "Labels": "28 emotions → mapped to 6"},
    "DistilBERT (GoEmotions)":    {"Architecture": "DistilBERT-base", "Training data": "GoEmotions (58k Reddit comments)", "Labels": "28 emotions → mapped to 6"},
    "RoBERTa (DAIR-AI)":          {"Architecture": "RoBERTa-base", "Training data": "DAIR-AI emotion dataset", "Labels": "6 emotions"},
    "DistilBERT (DAIR-AI)":       {"Architecture": "DistilBERT-base", "Training data": "DAIR-AI emotion dataset", "Labels": "6 emotions"},
    "Mistral 7B Instruct (LLM)":  {"Architecture": "Mistral-7B", "Training data": "Instruction-tuned on diverse data", "Labels": "Zero-shot via logit scoring"},
    "Phi-3 Mini 4k Instruct (LLM)":{"Architecture": "Phi-3-mini-4k", "Training data": "Instruction-tuned on diverse data", "Labels": "Zero-shot via logit scoring"},
}

# Confidence threshold for uncertain predictions
UNCERTAINTY_THRESHOLD = st.sidebar.slider(
    "Uncertainty threshold", min_value=0.1, max_value=0.9, value=0.4, step=0.05,
    help="Rows where max predicted probability is below this are flagged as uncertain."
)

# ---------------- PROMPT ----------------
def build_zero_shot_prompt(text, multi_label=False):
    emotion_list = ", ".join(EMOTIONS)
    if not multi_label:

        return f"""You are an expert emotion classifier.

        Choose the ONE emotion that best describes the text.

        {emotion_list}

        Respond using only one emotion name.

        Text: "{text}"
        Emotion:"""

    else:
        return f"""Classify the emotions in the text below.

        The text may express multiple emotions.
        Available emotions: {emotion_list}

        Text: "{text}"

        Give the emotions (comma separated):"""

# ---------------- MODEL LOADING ----------------
@st.cache_resource
def load_model(choice):
    if choice in HF_MODELS:
        tokenizer = AutoTokenizer.from_pretrained(HF_MODELS[choice])
        model = AutoModelForSequenceClassification.from_pretrained(HF_MODELS[choice])
        model = model.to(device)
        model_type = "bert"

    elif "Mistral" in choice:
        tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2")
        model = AutoModelForCausalLM.from_pretrained(
            "mistralai/Mistral-7B-Instruct-v0.2",
            device_map="auto",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        )
        model_type = "llm"

    else:
        tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
        model = AutoModelForCausalLM.from_pretrained(
            "microsoft/Phi-3-mini-4k-instruct",
            device_map="auto",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
        )
        model_type = "llm"

    return tokenizer, model, model_type

# ---------------- BERT PREDICTION ----------------
def predict_bert(tokenizer, model, text, model_choice):
    dev = next(model.parameters()).device
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(dev) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
    raw_labels = [model.config.id2label[i] for i in range(model.config.num_labels)]

    # Aggregate GoEmotions 28-class output into 6 target emotions
    if "GoEmotions" in model_choice:
        probs, labels = aggregate_goemotions(probs, raw_labels)
    else:
        labels = raw_labels

    return probs, labels

# ---------------- LLM LOGIT SCORING ----------------
def score_labels_cached(prompt, tokenizer, model, device, label_token_ids):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs, use_cache=True)

    base_past      = outputs.past_key_values
    base_logits    = outputs.logits[:, -1, :]
    base_log_probs = F.log_softmax(base_logits, dim=-1)

    scores = {}
    for label, token_ids in label_token_ids.items():
        total_log_prob = 0.0
        past      = base_past
        log_probs = base_log_probs

        for i, token_id in enumerate(token_ids):
            total_log_prob += log_probs[0, token_id].item()

            if i < len(token_ids) - 1:
                next_token = torch.tensor([[token_id]], device=device)
                with torch.no_grad():
                    out = model(input_ids=next_token, past_key_values=past, use_cache=True)
                past      = out.past_key_values
                logits    = out.logits[:, -1, :]
                log_probs = F.log_softmax(logits, dim=-1)

        scores[label] = total_log_prob   # summed

    return scores

# ---------------- LLM DEVICE HELPER ----------------
def get_llm_device(model):
    """
    FIX 5: With device_map='auto', layers may be on different devices.
    Safely retrieve the input device by checking the first parameter's device,
    but fall back to cuda:0 / cpu based on availability.
    """
    try:
        return next(model.parameters()).device
    except StopIteration:
        pass
    return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# ---------------- LLM PREDICTION ----------------
def predict_llm_single(text, tokenizer, model, multi_label=False, label_token_ids=None):
    dev = get_llm_device(model)

    if label_token_ids is None:
        label_token_ids = {
            label: tokenizer(label, add_special_tokens=False)["input_ids"]
            for label in EMOTIONS
        }

    prompt = build_zero_shot_prompt(text, multi_label)
    scores = score_labels_cached(prompt, tokenizer, model, dev, label_token_ids)

    # Convert raw summed log-prob scores to a softmax probability distribution
    score_vals = np.array([scores[e] for e in EMOTIONS], dtype=np.float64)
    score_vals -= score_vals.max()
    probs = np.exp(score_vals) / np.exp(score_vals).sum()

    return probs, EMOTIONS

# ---------------- METRICS ----------------
def compute_metrics(y_true, y_pred, labels):
    label2id = {l: i for i, l in enumerate(labels)}


    pairs = [
        (label2id[t], label2id[p])
        for t, p in zip(y_true, y_pred)
        if t in label2id and p in label2id
    ]
    if not pairs:
        empty_df = pd.DataFrame({"Metric": [], "Score": []})
        per_df   = pd.DataFrame({"Emotion": labels, "F1 Score": [0.0] * len(labels)})
        cm_arr   = np.zeros((len(labels), len(labels)), dtype=int)
        return empty_df, per_df, cm_arr

    y_true_ids, y_pred_ids = zip(*pairs)
    y_true_ids = list(y_true_ids)
    y_pred_ids = list(y_pred_ids)

    overall_df = pd.DataFrame({
        "Metric": ["Precision (Macro)", "Recall (Macro)", "F1 (Macro)", "F1 (Micro)", "MCC"],
        "Score":  [
            precision_score(y_true_ids, y_pred_ids, average="macro",  zero_division=0),
            recall_score   (y_true_ids, y_pred_ids, average="macro",  zero_division=0),
            f1_score       (y_true_ids, y_pred_ids, average="macro",  zero_division=0),
            f1_score       (y_true_ids, y_pred_ids, average="micro",  zero_division=0),
            matthews_corrcoef(y_true_ids, y_pred_ids),
        ]
    })
    overall_df["Score"] = overall_df["Score"].round(4)

    per_f1 = f1_score(y_true_ids, y_pred_ids, average=None,
                      labels=list(range(len(labels))), zero_division=0)
    per_df = pd.DataFrame({"Emotion": labels, "F1 Score": per_f1.round(4)}) \
               .sort_values("F1 Score", ascending=False)

    cm_arr = confusion_matrix(y_true_ids, y_pred_ids, labels=list(range(len(labels))))
    return overall_df, per_df, cm_arr

# ---------------- CONFUSION MATRIX PLOT ----------------
def plot_confusion_matrix(cm_arr, labels):
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(
        cm_arr, annot=True, fmt="d", cmap="Blues",
        xticklabels=labels, yticklabels=labels, ax=ax
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion Matrix")
    plt.tight_layout()
    return fig

# ---------------- CALIBRATION PLOT ----------------
def plot_calibration(probs_array, true_labels_list, pred_labels, all_labels, n_bins=10):
    """Reliability diagram: mean predicted confidence vs actual accuracy per bin."""
    confidences = probs_array.max(axis=1)
    correct     = np.array([t == p for t, p in zip(true_labels_list, pred_labels)]).astype(float)

    bins        = np.linspace(0, 1, n_bins + 1)
    bin_centers, bin_acc, bin_conf, bin_counts = [], [], [], []

    for i in range(n_bins):
        mask = (confidences >= bins[i]) & (confidences < bins[i + 1])
        if mask.sum() > 0:
            bin_centers.append((bins[i] + bins[i + 1]) / 2)
            bin_acc.append(correct[mask].mean())
            bin_conf.append(confidences[mask].mean())
            bin_counts.append(mask.sum())

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax.bar(bin_centers, bin_acc, width=0.08, alpha=0.5, color="#5B8DB8", label="Accuracy")
    ax.plot(bin_conf, bin_acc, "o-", color="#F3722C", label="Model")
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Accuracy")
    ax.set_title("Calibration Plot")
    ax.legend()
    plt.tight_layout()
    return fig

# ---------------- t-SNE PLOT ----------------
def plot_tsne(probs_array, color_labels, title="t-SNE of Probability Vectors"):
    if len(probs_array) < 5:
        return None
    perplexity = min(30, len(probs_array) - 1)
    tsne   = TSNE(n_components=2, perplexity=perplexity, random_state=42)
    coords = tsne.fit_transform(probs_array)

    unique = list(dict.fromkeys(color_labels))

    fig, ax = plt.subplots(figsize=(6, 5))
    for emotion in unique:
        idx = [i for i, l in enumerate(color_labels) if l == emotion]
        ax.scatter(
            coords[idx, 0], coords[idx, 1],
            c=EMOTION_COLORS.get(emotion, "#888888"),
            label=emotion, alpha=0.7, s=40
        )
    ax.set_title(title)
    ax.legend(loc="best", fontsize=7)
    plt.tight_layout()
    return fig

# ---------------- VIOLIN PLOT ----------------
def plot_violin(probs_array, labels):
    df_melt = pd.DataFrame(probs_array, columns=labels).melt(
        var_name="Emotion", value_name="Probability"
    )
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.violinplot(
        data=df_melt, x="Emotion", y="Probability",
        palette=[EMOTION_COLORS.get(e, "#888") for e in labels],
        inner="box", ax=ax,
        cut=0
    )
    ax.set_ylim(0, 1)
    ax.set_title("Probability Distribution per Emotion")
    plt.tight_layout()
    return fig

# ---------------- DOWNLOAD HELPER ----------------
def df_to_csv_bytes(df):
    buf = io.BytesIO()
    df.to_csv(buf, index=False)
    return buf.getvalue()


# UI

# ---- Sidebar: model cards ----
st.sidebar.markdown("---")
st.sidebar.subheader("📖 Model Cards")

example_texts = [
    "I am so happy today!",
    "This makes me furious!",
    "I feel really anxious about tomorrow.",
    "I can't stop smiling, life is great!",
    "I'm heartbroken and sad."
]

model_choices = st.multiselect(
    "Choose models:",
    list(HF_MODELS.keys()) + [
        "Mistral 7B Instruct (LLM)",
        "Phi-3 Mini 4k Instruct (LLM)"
    ],
    default=["RoBERTa (DAIR-AI)"]
)

for mc in model_choices:
    if mc in MODEL_CARDS:
        with st.sidebar.expander(mc):
            for k, v in MODEL_CARDS[mc].items():
                st.write(f"**{k}:** {v}")

llm_selected = [m for m in model_choices if "LLM" in m]
if len(llm_selected) > 1:
    st.error("⚠️ Only ONE LLM allowed.")
    st.stop()

# ---- TEXT INPUT ----
example_choice = st.selectbox("Pick example (optional):", [""] + example_texts)
text_input     = st.text_area("Enter text:", value=example_choice if example_choice else "")

# Text length warning
if text_input:
    n_tokens_approx = len(text_input.split())
    if n_tokens_approx > 200:
        st.warning(f"⚠️ Text is ~{n_tokens_approx} words — very long inputs may be truncated by the model.")
    elif n_tokens_approx < 3:
        st.warning("⚠️ Text is very short — predictions may be unreliable.")

multi_label = st.checkbox("Use multi-label (LLM only)")

# ---- SINGLE TEXT BUTTON ----
single_text_clicked = st.button("Classify Single Text")

# ---------------- SINGLE TEXT ----------------
if single_text_clicked and text_input:
    tabs = st.tabs(model_choices)

    for i, model_choice in enumerate(model_choices):
        with tabs[i]:
            tokenizer, model, model_type = load_model(model_choice)

            with st.spinner(f"Running {model_choice}..."):
                if model_type == "bert":
                    probs, labels = predict_bert(tokenizer, model, text_input, model_choice)
                else:
                    probs, labels = predict_llm_single(
                        text_input, tokenizer, model, multi_label
                    )

            df = pd.DataFrame({
                "Emotion": labels,
                "Probability": probs
            }).sort_values(by="Probability", ascending=False)

            # Uncertainty flag
            max_conf = probs.max()
            if max_conf < UNCERTAINTY_THRESHOLD:
                st.warning(f"⚠️ Low confidence ({max_conf:.2f}) — prediction may be unreliable.")

            st.write(df.head(3))

            fig, ax = plt.subplots()
            bar_colors = [EMOTION_COLORS.get(e, "#888") for e in df["Emotion"]]
            ax.barh(df["Emotion"], df["Probability"], color=bar_colors)
            ax.set_xlabel("Probability")
            ax.invert_yaxis()
            ax.set_title(f"{model_choice} — Top Emotions")
            st.pyplot(fig)


# ---------------- CSV ----------------
st.divider()
st.subheader("📂 Upload CSV for Batch Analysis")

uploaded_file = st.file_uploader("Upload CSV", type=["csv"])

if uploaded_file:
    df_csv = pd.read_csv(uploaded_file)

    if "text" not in df_csv.columns:
        st.error("CSV must contain 'text' column")
    else:
        text_list  = df_csv["text"].tolist()
        has_labels = "emotion" in df_csv.columns

        # ---- Dataset summary ----
        st.subheader("📋 Dataset Summary")
        st.dataframe(pd.DataFrame({"Total rows": [len(df_csv)]}), use_container_width=False)

        if has_labels:
            df_csv["emotion"] = df_csv["emotion"].str.lower().str.strip()
            emotion_counts = df_csv["emotion"].value_counts().reset_index()
            emotion_counts.columns = ["Emotion", "Count"]
            st.write("**Examples per emotion:**")
            st.dataframe(emotion_counts, use_container_width=False)

        batch_model_choice = st.selectbox("Model for batch:", model_choices)
        csv_batch_clicked  = st.button("Run CSV Batch")

        if csv_batch_clicked:
            tokenizer, model, model_type = load_model(batch_model_choice)

            # Pre-compute label token ids once for LLMs
            label_token_ids = None
            if model_type == "llm":
                label_token_ids = {
                    label: tokenizer(label, add_special_tokens=False)["input_ids"]
                    for label in EMOTIONS
                }

            probs_list       = []
            pred_labels_list = []
            labels           = None
            progress_bar     = st.progress(0)

            with st.spinner("Running batch predictions..."):
                for idx, text in enumerate(text_list):
                    if model_type == "bert":
                        probs, labels = predict_bert(tokenizer, model, text, batch_model_choice)
                    else:
                        probs, labels = predict_llm_single(
                            text, tokenizer, model, multi_label, label_token_ids
                        )

                    probs_list.append(probs)
                    pred_labels_list.append(labels[int(np.argmax(probs))])
                    progress_bar.progress((idx + 1) / len(text_list))

            probs_array = np.array(probs_list)
            pred_dist_df = (
                pd.DataFrame(probs_array, columns=labels)
                .mean()
                .sort_values(ascending=False)
                .reset_index()
            )
            pred_dist_df.columns = ["Emotion", "Average Probability"]

            # Uncertain rows
            max_confs    = probs_array.max(axis=1)
            uncertain_mask = max_confs < UNCERTAINTY_THRESHOLD
            n_uncertain  = uncertain_mask.sum()
            if n_uncertain > 0:
                st.warning(f"⚠️ {n_uncertain} row(s) had max confidence < {UNCERTAINTY_THRESHOLD} and may be unreliable.")

            # Build full results dataframe (used for download)
            results_df = df_csv.copy()
            results_df["predicted_emotion"] = pred_labels_list
            results_df["confidence"]        = max_confs.round(4)
            for e, col in zip(labels, probs_array.T):
                results_df[f"prob_{e}"] = col.round(4)
            results_df["uncertain"] = uncertain_mask

            # ---- Distribution ----
            st.subheader("📊 Emotion Distribution")

            if has_labels:
                true_counts  = df_csv["emotion"].value_counts()
                true_dist_df = pd.DataFrame({
                    "Emotion":    true_counts.index,
                    "Proportion": (true_counts.values / true_counts.sum()).round(4)
                })

                col1, col2 = st.columns(2)
                with col1:
                    st.write("**Model Predictions**")
                    st.dataframe(pred_dist_df)
                    fig1, ax1 = plt.subplots()
                    bar_colors = [EMOTION_COLORS.get(e, "#888") for e in pred_dist_df["Emotion"]]
                    ax1.barh(pred_dist_df["Emotion"], pred_dist_df["Average Probability"], color=bar_colors)
                    ax1.invert_yaxis()
                    ax1.set_title("Predicted Distribution")
                    st.pyplot(fig1)

                with col2:
                    st.write("**Actual Labels**")
                    st.dataframe(true_dist_df)
                    fig2, ax2 = plt.subplots()
                    bar_colors2 = [EMOTION_COLORS.get(e, "#888") for e in true_dist_df["Emotion"]]
                    ax2.barh(true_dist_df["Emotion"], true_dist_df["Proportion"], color=bar_colors2)
                    ax2.invert_yaxis()
                    ax2.set_title("Actual Distribution")
                    st.pyplot(fig2)

                # ---- Metrics ----
                st.subheader("📈 Evaluation Metrics")
                true_labels_list = df_csv["emotion"].tolist()
                overall_df, per_df, cm_arr = compute_metrics(
                    true_labels_list, pred_labels_list, labels
                )

                mcol1, mcol2 = st.columns(2)
                with mcol1:
                    st.write("**Overall Metrics**")
                    st.dataframe(overall_df, use_container_width=True)
                with mcol2:
                    st.write("**Per-Emotion F1**")
                    st.dataframe(per_df, use_container_width=True)

                # ---- Confusion matrix ----
                st.subheader("Confusion Matrix")
                fig_cm = plot_confusion_matrix(cm_arr, EMOTIONS)
                st.pyplot(fig_cm)

                # ---- Calibration (BERT only — LLM scores are not true probabilities) ----
                if model_type != "llm":
                    st.subheader("Confidence Calibration")
                    fig_cal = plot_calibration(probs_array, true_labels_list, pred_labels_list, EMOTIONS)
                    st.pyplot(fig_cal)
                else:
                    st.info("ℹ️ Calibration plot is not shown for LLM models — their scores are normalised log-probs, not true probabilities.")

                # ---- Error analysis ----
                st.subheader("🔍 Error Analysis")
                error_df = results_df[results_df["predicted_emotion"] != results_df["emotion"]] \
                    [["text", "emotion", "predicted_emotion", "confidence"]] \
                    .sort_values("confidence", ascending=False)
                st.write(f"**{len(error_df)} misclassified rows** (sorted by model confidence)")
                st.dataframe(error_df, use_container_width=True)

            else:
                # No labels — predictions only
                st.dataframe(pred_dist_df)
                fig, ax = plt.subplots()
                bar_colors = [EMOTION_COLORS.get(e, "#888") for e in pred_dist_df["Emotion"]]
                ax.barh(pred_dist_df["Emotion"], pred_dist_df["Average Probability"], color=bar_colors)
                ax.invert_yaxis()
                ax.set_title("Predicted Distribution")
                st.pyplot(fig)

            # ---- Violin plot (BERT only — LLM scores are not true probabilities) ----
            if model_type != "llm":
                st.subheader("Probability Spread per Emotion")
                fig_vio = plot_violin(probs_array, labels)
                st.pyplot(fig_vio)
            else:
                st.info("ℹ️ Violin plot is not shown for LLM models — their scores are normalised log-probs, not true probabilities.")

            st.subheader("t-SNE of Probability Vectors")
            fig_tsne = plot_tsne(probs_array, pred_labels_list,
                                  title=f"t-SNE — {batch_model_choice}")
            if fig_tsne:
                st.pyplot(fig_tsne)
            else:
                st.info("Need at least 5 rows for t-SNE.")

            # ---- Download ----
            st.subheader("⬇️ Download Predictions")
            st.download_button(
                label="Download predictions CSV",
                data=df_to_csv_bytes(results_df),
                file_name="predictions.csv",
                mime="text/csv"
            )

Writing app.py


##Run Application in Localhost

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3BY0Q7ZfVbzqZasHZzs1rm3kV9e_dLu12sjtYfZht613Hpgi")

!streamlit run app.py &>/content/logs.txt &

public_url = ngrok.connect(8501)
print(public_url)

!streamlit run app.py